# Providers 03 - vLLM + Unsloth + Qwen

Objetivo: usar este notebook como punto de entrada único de Colab para vllm-runtime. Recupera la receta de Agentic Systems 1.0, pero usa la API pública 2.1: ModelArtifact, ModelServer, RuntimeConfig, Agent, RunResult y la certificación de los cuatro Frameworks.

**Lugar en el modelo:** vLLM es el Provider de inferencia y ModelServer es la frontera de infraestructura; Unsloth/Qwen identifica el artefacto, mientras Agent y System conservan su lógica.

**Evidencia exigida:** live debe producir un RunResult con engine vllm-runtime, ejecutar multiply y emitir una attestation aprobada para native, LangGraph, OpenAI Agents y Strands.

**Límite de la evidencia:** la ruta offline sólo demuestra construcción, schemas y not-run. La compatibilidad GPU exige ejecutar la ruta live sobre el wheel y commit exactos.

El servidor tiene ciclo de vida explícito. Construirlo no inicia procesos; start registra endpoint y PID, y stop termina únicamente el proceso propio.

Requisito live: Runtime de Colab con GPU. Para certificar un release, sube el wheel candidato exacto y completa COMMIT_SHA con el commit que produjo ese wheel. RUN_VLLM_LIVE=0 conserva una ruta offline not-run para los gates.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_VLLM_LIVE | 1 | Usa 0 para ejecutar sólo el contrato offline. |
| COMMIT_SHA | candidato 2.1.0 | Commit exacto que produjo el wheel candidato. |
| EXPECTED_WHEEL_SHA256 | candidato 2.1.0 | Hash que debe tener el wheel subido. |
| MODEL_ID | unsloth/Qwen3-0.6B | Modelo ligero con tool calling para Colab. |
| PROFILE | auto | Resuelve fast, medium o power desde VRAM; admite override. |
| VLLM_DTYPE | automático | half en T4; bfloat16 desde compute capability 8.0. |
| VLLM_ENABLE_THINKING | 0 | Desactiva reasoning en tool calling multi-turn. |
| VLLM_TEMPERATURE | 0.7 | Sampling recomendado por Qwen3 en modo non-thinking. |

En Colab: selecciona GPU, ejecuta en orden y sube el wheel indicado. Los valores
pueden sustituirse mediante variables de entorno para certificar otro candidato.

## Contrato de la demostracion

La misma fachada pública cubre el ciclo completo: model_artifact, model_server, runtime, system, agent y RunResult. El runner final sustituye sólo el Framework y conserva Provider, endpoint y modelo.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path


def load_canonical_dotenv(start: Path) -> Path | None:
    # The explicit path exists for isolated CI; otherwise the nearest .env is king.
    explicit = os.getenv("AGENTIC_SYSTEMS_DOTENV")
    candidates = (
        (Path(explicit).expanduser().resolve(),)
        if explicit
        else tuple(directory / ".env" for directory in (start.resolve(), *start.resolve().parents))
    )
    for candidate in candidates:
        if not candidate.is_file():
            continue
        for raw_line in candidate.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            if key:
                os.environ[key] = value.strip().strip('"').strip("'")
        return candidate
    return None


DOTENV_PATH = load_canonical_dotenv(Path.cwd())
RUN_VLLM_LIVE = os.getenv("RUN_VLLM_LIVE", "1").strip().lower() in {"1", "true", "yes"}
COMMIT_SHA = os.getenv("AGENTIC_SYSTEMS_COMMIT_SHA", "").strip()
EXPECTED_WHEEL_FILENAME = os.getenv("AGENTIC_SYSTEMS_WHEEL_FILENAME", "").strip()
EXPECTED_WHEEL_SHA256 = os.getenv("AGENTIC_SYSTEMS_WHEEL_SHA256", "").strip().lower()
MODEL_ID = os.getenv("VLLM_MODEL", "unsloth/Qwen3-0.6B")
BASE_MODEL_ID = os.getenv("VLLM_BASE_MODEL", "Qwen/Qwen3-0.6B")
REQUESTED_PROFILE = os.getenv("VLLM_PROFILE", "fast").strip().lower()
VLLM_HOST = os.getenv("VLLM_HOST", "127.0.0.1")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", f"http://{VLLM_HOST}:{VLLM_PORT}/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "vllm")
VLLM_TOOL_CALL_PARSER = os.getenv("VLLM_TOOL_CALL_PARSER", "hermes")
VLLM_ENABLE_THINKING = os.getenv("VLLM_ENABLE_THINKING", "0").strip().lower() in {
    "1", "true", "yes"
}
VLLM_REASONING_PARSER = os.getenv(
    "VLLM_REASONING_PARSER", "qwen3" if VLLM_ENABLE_THINKING else ""
).strip() or None
VLLM_TEMPERATURE = float(os.getenv("VLLM_TEMPERATURE", "0.7"))
VLLM_GPU_MEMORY_UTILIZATION = float(os.getenv("VLLM_GPU_MEMORY_UTILIZATION", "0.4"))
VLLM_MAX_MODEL_LEN = int(os.getenv("VLLM_MAX_MODEL_LEN", "2048"))
VLLM_MAX_NUM_SEQS = int(os.getenv("VLLM_MAX_NUM_SEQS", "4"))
assert REQUESTED_PROFILE in {"auto", "fast", "medium", "power", "custom"}

WHEEL_PATH = None
if RUN_VLLM_LIVE:
    assert len(COMMIT_SHA) == 40, "Define AGENTIC_SYSTEMS_COMMIT_SHA en .env"
    assert EXPECTED_WHEEL_FILENAME.endswith(".whl"), (
        "Define AGENTIC_SYSTEMS_WHEEL_FILENAME en .env"
    )
    assert len(EXPECTED_WHEEL_SHA256) == 64, (
        "Define AGENTIC_SYSTEMS_WHEEL_SHA256 en .env"
    )
    configured_wheel = os.getenv("AGENTIC_SYSTEMS_WHEEL")
    search_roots = [Path.cwd(), Path("/content")]
    wheel_candidates = (
        [Path(configured_wheel).expanduser()]
        if configured_wheel
        else [
            root / EXPECTED_WHEEL_FILENAME
            for root in search_roots
            if (root / EXPECTED_WHEEL_FILENAME).is_file()
        ]
    )
    wheel_candidates = list(dict.fromkeys(path.resolve() for path in wheel_candidates))
    if len(wheel_candidates) > 1:
        raise RuntimeError(
            "Existe más de un wheel candidato; define AGENTIC_SYSTEMS_WHEEL en .env"
        )
    if wheel_candidates:
        WHEEL_PATH = str(wheel_candidates[0])
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(
                f"Coloca {EXPECTED_WHEEL_FILENAME} junto al notebook"
            ) from exc
        uploaded = files.upload()
        wheel_names = [name for name in uploaded if name.endswith(".whl")]
        assert len(wheel_names) == 1, "Sube exactamente un wheel de agentic-systems"
        WHEEL_PATH = "/content/" + wheel_names[0]

## 1) Instalar el wheel y vLLM

La instalación usa el backend de Torch resuelto por vLLM para no mezclar wheels CUDA. TorchVision se instala en la misma transacción porque vLLM 0.27 lo importa durante el warm-up incluso para modelos de texto; sólo TorchAudio se retira si Colab dejó una variante CUDA incompatible.

In [ ]:
if RUN_VLLM_LIVE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
    subprocess.run([
        "uv", "pip", "install", "-U", "vllm", "torchvision",
        "--torch-backend=auto",
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps", WHEEL_PATH,
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "openai>=2.45,<3", "openai-agents>=0.18.3,<0.19",
        "langgraph>=0.2", "strands-agents>=1.29,<2", "mcp>=1,<2",
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"
    ], check=False)

In [ ]:
import hashlib
import platform
from importlib.metadata import version as package_version

import agentic_systems as toolkit

wheel_sha256 = hashlib.sha256(Path(WHEEL_PATH).read_bytes()).hexdigest() if WHEEL_PATH else None
environment = {
    "package": toolkit.__name__,
    "version": toolkit.__version__,
    "python": platform.python_version(),
    "run_live": RUN_VLLM_LIVE,
    "wheel_sha256": wheel_sha256,
    "dotenv": str(DOTENV_PATH) if DOTENV_PATH else None,
    "model": MODEL_ID,
    "requested_profile": REQUESTED_PROFILE,
    "host": VLLM_HOST,
    "port": VLLM_PORT,
    "api_key_configured": bool(VLLM_API_KEY),
}
if RUN_VLLM_LIVE:
    import torch

    assert Path(WHEEL_PATH).name == EXPECTED_WHEEL_FILENAME, (
        Path(WHEEL_PATH).name,
        EXPECTED_WHEEL_FILENAME,
    )
    assert wheel_sha256 == EXPECTED_WHEEL_SHA256, (
        wheel_sha256,
        EXPECTED_WHEEL_SHA256,
    )
    environment.update(
        torch=torch.__version__,
        torchvision=package_version("torchvision"),
        vllm=package_version("vllm"),
        cuda_available=torch.cuda.is_available(),
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        cuda_capability=(
            list(torch.cuda.get_device_capability(0))
            if torch.cuda.is_available()
            else None
        ),
    )
    assert toolkit.__version__ == "2.1.0"
    assert torch.cuda.is_available(), "Selecciona Runtime > Change runtime type > GPU"

    cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
    gpu_properties = torch.cuda.get_device_properties(0)
    gpu_memory_gib = round(gpu_properties.total_memory / (1024 ** 3), 2)
    VLLM_DTYPE = os.getenv("VLLM_DTYPE") or (
        "half" if cuda_major < 8 else "bfloat16"
    )
    if REQUESTED_PROFILE == "auto":
        PROFILE = (
            "fast"
            if gpu_memory_gib < 20
            else "medium"
            if gpu_memory_gib < 60
            else "power"
        )
    else:
        PROFILE = REQUESTED_PROFILE
    SERVER_EXTRA_ARGS = (
        "--dtype", VLLM_DTYPE,
        "--default-chat-template-kwargs",
        json.dumps({"enable_thinking": VLLM_ENABLE_THINKING}),
    )
    environment.update(
        requested_vllm_profile=REQUESTED_PROFILE,
        selected_vllm_profile=PROFILE,
        vllm_dtype=VLLM_DTYPE,
        cuda_capability=[cuda_major, cuda_minor],
        gpu_memory_gib=gpu_memory_gib,
    )
else:
    PROFILE = REQUESTED_PROFILE if REQUESTED_PROFILE != "auto" else "fast"
    VLLM_DTYPE = None
    SERVER_EXTRA_ARGS = ()
toolkit.show_json(environment, title="Preflight vLLM")

## 2) Declarar artefacto, servidor y runtime

ModelArtifact separa la identidad servida de su modelo base. Esa frontera permite sustituir MODEL_ID por un checkpoint fine-tuned o un adapter LoRA sin rediseñar el sistema agéntico.

In [ ]:
artifact = toolkit.model_artifact(
    MODEL_ID,
    base_model=BASE_MODEL_ID,
    metadata={"serving_library": "unsloth"},
)
server = toolkit.model_server(
    artifact,
    backend="vllm",
    profile=PROFILE,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    host=VLLM_HOST,
    port=VLLM_PORT,
    tool_call_parser=VLLM_TOOL_CALL_PARSER,
    reasoning_parser=VLLM_REASONING_PARSER,
    extra_args=SERVER_EXTRA_ARGS,
    startup_timeout_s=600,
    log_path="/content/vllm-server.log",
)
toolkit.show_json(server.inspect(), title="ModelServer declarado")

In [ ]:
if RUN_VLLM_LIVE:
    try:
        endpoint = server.start()
    except Exception as exc:
        log_path = Path("/content/vllm-server.log")
        log_text = (
            log_path.read_text(encoding="utf-8", errors="replace")
            if log_path.exists()
            else ""
        )
        diagnostic_lines = [
            line
            for line in log_text.splitlines()
            if any(
                token in line.lower()
                for token in (
                    "error",
                    "exception",
                    "runtimeerror",
                    "valueerror",
                    "traceback",
                    "cuda",
                    "bfloat16",
                    "dtype",
                )
            )
        ]
        toolkit.show_json(
            {
                "error_type": type(exc).__name__,
                "message": str(exc),
                "profile": PROFILE,
                "dtype": VLLM_DTYPE,
                "log_head": log_text[:24000],
                "log_tail": log_text[-24000:],
                "diagnostic_lines": diagnostic_lines[-200:],
            },
            title="vLLM startup failure",
        )
        raise
    health = server.health()
    toolkit.show_json(health.model_dump(mode="json"), title="vLLM health")
    assert health.status == "healthy", "Revisa /content/vllm-server.log"
    runtime = server.runtime(metadata={"tutorial": "providers/vllm-unsloth-qwen"})
else:
    endpoint = None
    health = None
    runtime = toolkit.runtime(
        provider="vllm-runtime",
        model=MODEL_ID,
        endpoint=VLLM_BASE_URL,
        metadata={"tutorial": "providers/vllm-unsloth-qwen", "live": False},
    )
toolkit.show_json(runtime.describe(), title="vLLM RuntimeConfig")

## 3) Crear system, tool y agent

Esta prueba valida el camino relevante: el modelo solicita una Tool, Agentic Systems la ejecuta y RunResult conserva la misma proyección pública que los demás Providers.

In [ ]:
@toolkit.tool
def multiply(a: int, b: int) -> dict:
    return {"result": a * b}

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name="qwen_calculator",
    instructions="Usa multiply para calcular. Devuelve sólo el resultado.",
    tools=[multiply],
    contract=toolkit.AgentContract(
        must_call=["multiply"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=VLLM_TEMPERATURE,
        tool_choice="multiply",
    ),
)
toolkit.show_json(agent.info(), title="Agente declarado")

In [ ]:
if RUN_VLLM_LIVE:
    result = agent.run("¿Cuánto es 17 por 19?", mode="eval")
    assert isinstance(result, toolkit.RunResult)
    assert result.ok, result.errors
    assert result.engine == "vllm-runtime"
    assert [event.name for event in result.tool_events] == ["multiply"]
    result.check_invariants()
    toolkit.human_result(result, title="vLLM RunResult", show_lineage=True)
    toolkit.show_json(toolkit.run_result_output(result), title="Contrato normalizado")
else:
    result = None
    toolkit.show_json({
        "status": "not-run",
        "provider": "vllm-runtime",
        "reason": "RUN_VLLM_LIVE=0; no se inició infraestructura GPU.",
    }, title="vLLM live gate")

## 4) Certificar los cuatro frameworks

El runner oficial consume el mismo endpoint. La attestation compara invariantes, identidad, Tools, errores y round-trip; no exige texto idéntico entre Frameworks.

In [ ]:
if RUN_VLLM_LIVE:
    if not COMMIT_SHA:
        raise ValueError("Completa COMMIT_SHA con el commit exacto que produjo el wheel")
    local_runner = Path.cwd() / "run_live_matrix.py"
    if local_runner.is_file():
        matrix_runner = local_runner
    else:
        repo = Path("/content/agentic-systems")
        if not repo.exists():
            subprocess.run([
                "git", "clone", "https://github.com/JacoboGGLeon/agentic_systems.git", str(repo)
            ], check=True)
        subprocess.run(["git", "-C", str(repo), "fetch", "--all", "--tags"], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", COMMIT_SHA], check=True)
        matrix_runner = repo / "scripts" / "run_live_matrix.py"
    os.environ.update(
        VLLM_BASE_URL=endpoint.base_url,
        VLLM_API_KEY=VLLM_API_KEY,
        VLLM_MODEL=artifact.model_id,
    )
    OUTPUT = Path("/content/vllm-attestation.json")
    completed = subprocess.run([
        sys.executable, str(matrix_runner),
        "--wheel", WHEEL_PATH,
        "--output", str(OUTPUT),
        "--commit", COMMIT_SHA,
        "--providers", "vllm-runtime",
        "--frameworks", "native", "langgraph", "openai-agents", "strands",
    ], text=True, capture_output=True)
    toolkit.show_json({
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }, title="vLLM live matrix")
    if completed.returncode:
        toolkit.show_json({
            "vllm_log_tail": Path("/content/vllm-server.log").read_text(errors="replace")[-12000:],
        }, title="Diagnóstico vLLM")
        raise RuntimeError(f"La matriz live falló con código {completed.returncode}")
    attestation = json.loads(OUTPUT.read_text())
    failed_cases = [case for case in attestation["cases"] if not case["ok"]]
    summary = {
        "total": len(attestation["cases"]),
        "passed": len(attestation["cases"]) - len(failed_cases),
        "failed": len(failed_cases),
    }
    toolkit.show_json(summary, title="Attestation summary")
    assert attestation["wheel_sha256"] == wheel_sha256
    assert attestation["commit_sha"] == COMMIT_SHA
    assert summary["total"] == 4
    assert summary["failed"] == 0
    files.download(str(OUTPUT))
else:
    toolkit.show_json({"status": "not-run", "scope": "vllm-attestation"}, title="Attestation gate")

## 5) API realmente ejercitada

La cobertura enumera sólo llamadas presentes en las celdas anteriores. Servir el modelo y consumirlo son contratos separados, conectados por RuntimeConfig.

In [ ]:
api_coverage = [
    "toolkit.model_artifact",
    "toolkit.model_server",
    "server.inspect",
    "server.start",
    "server.health",
    "server.runtime",
    "server.stop",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.RunResult",
    "toolkit.show_json",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
]
toolkit.show_json(api_coverage, title="vLLM API coverage")

## Resultado e interpretacion

Con live desactivado: contratos declarativos y estados not-run ejecutables. Con live activado: un RunResult vllm-runtime real y una attestation de native, LangGraph, OpenAI Agents y Strands. Para fine-tuning, entrena y guarda un checkpoint o adapter con Unsloth, cambia MODEL_ID y conserva el resto.

In [ ]:
if RUN_VLLM_LIVE:
    server.stop()
    toolkit.show_json(
        {"status": "stopped", "owned_process_only": True},
        title="ModelServer closure",
    )